# Weather Silver Layer

Transforms Open-Meteo Forecast API responses from `weather_api.bronze.weather_raw` into clean, typed, analytics-ready Silver tables.

**Tables created:**
- `weather_api.silver.weather_current` — One row per location per API snapshot (grain: `location_name` + `snapshot_at`)
- `weather_api.silver.weather_hourly` — One row per location, forecast hour, and API snapshot (grain: `location_name` + `snapshot_at` + `forecast_for`)
- `weather_api.silver.weather_daily` — One row per location, forecast date, and API snapshot (grain: `location_name` + `snapshot_at` + `forecast_date`)

**Units:** Temperature in °C, humidity in %, pressure in hPa, wind speed/gusts in km/h, wind direction in °, precipitation/rain in mm, snowfall in cm, radiation in W/m², sunshine/daylight in s, visibility in m, soil moisture in m³/m³, vapour pressure deficit in kPa.

**Processing:** Batch read from Bronze → transform → Delta MERGE into Silver. Idempotent and safe for reruns. Forecast snapshot history is preserved — a forecast for the same hour can change between API retrievals.

In [0]:
# ════════════════════════════════════════════════════════════════════
# WEATHER SILVER — TRANSFORM
# ════════════════════════════════════════════════════════════════════
from pyspark.sql.functions import (
    col, explode, to_timestamp, to_date, arrays_zip,
    count as cnt, countDistinct, desc, max as max_fn, min as min_fn, lit, when
)
from delta.tables import DeltaTable

# ─── Schema & Config ────────────────────────────────────────────────
spark.sql("CREATE SCHEMA IF NOT EXISTS weather_api.silver")

BRONZE_TABLE     = "weather_api.bronze.weather_raw"
SILVER_CURRENT  = "weather_api.silver.weather_current"
SILVER_HOURLY   = "weather_api.silver.weather_hourly"
SILVER_DAILY    = "weather_api.silver.weather_daily"

# ─── Read Bronze Source ────────────────────────────────────────────
bronze = spark.table(BRONZE_TABLE)
total_bronze = bronze.count()
print(f"Bronze rows: {total_bronze}")

# ── Location ID lineage (backward-compatible) ──
# Check if metadata.location_id exists in the Bronze schema.  If not
# (historical data before registry migration), use NULL as a placeholder.
_metadata_fields = [f for f in bronze.schema.fields if f.name == "metadata"][0].dataType.fields
_has_location_id = "location_id" in [f.name for f in _metadata_fields]
if _has_location_id:
    _location_id_expr = col("metadata.location_id")
else:
    _location_id_expr = lit(None).cast("long").alias("location_id")
    print("⚠️  metadata.location_id not found in Bronze schema — using NULL placeholder")

# ─── Transform: Current Observations ───────────────────────────────
silver_current = bronze.select(
    # Lineage
    col("metadata.location_name"),
    _location_id_expr,
    col("metadata.latitude"),
    col("metadata.longitude"),
    col("raw_response.elevation").alias("elevation_m"),
    to_timestamp(col("metadata.retrieval_timestamp_utc")).alias("snapshot_at"),
    to_timestamp(col("raw_response.current.time")).alias("observed_at"),
    col("metadata.source"),
    col("metadata.api_endpoint"),
    col("_source_file_path").alias("source_file_path"),
    col("_ingest_timestamp").alias("ingested_at"),
    col("_rescued_data"),
    # Current weather measurements
    col("raw_response.current.temperature_2m"),
    col("raw_response.current.apparent_temperature"),
    col("raw_response.current.relative_humidity_2m").cast("int").alias("relative_humidity_2m"),
    col("raw_response.current.is_day").cast("boolean").alias("is_day"),
    col("raw_response.current.precipitation"),
    col("raw_response.current.rain"),
    col("raw_response.current.snowfall"),
    col("raw_response.current.weather_code").cast("int").alias("weather_code"),
    col("raw_response.current.cloud_cover").cast("int").alias("cloud_cover"),
    col("raw_response.current.pressure_msl"),
    col("raw_response.current.surface_pressure"),
    col("raw_response.current.wind_speed_10m"),
    col("raw_response.current.wind_direction_10m").cast("int").alias("wind_direction_10m"),
    col("raw_response.current.wind_gusts_10m"),
    # Measurement interval (seconds the current reading represents)
    col("raw_response.current.interval").cast("int").alias("interval_seconds"),
)
current_count = silver_current.count()
print(f"Current rows: {current_count}")

# ─── Transform: Hourly Forecast ────────────────────────────────────
HOURLY_SPECS = [
    ("time",                      "forecast_for",                "timestamp"),
    ("temperature_2m",            "temperature_2m",            "double"),
    ("relative_humidity_2m",      "relative_humidity_2m",      "int"),
    ("apparent_temperature",      "apparent_temperature",       "double"),
    ("precipitation",             "precipitation",             "double"),
    ("rain",                      "rain",                      "double"),
    ("snowfall",                  "snowfall",                  "double"),
    ("snow_depth",                "snow_depth",                "double"),
    ("weather_code",             "weather_code",             "int"),
    ("pressure_msl",             "pressure_msl",             "double"),
    ("surface_pressure",         "surface_pressure",         "double"),
    ("cloud_cover",              "cloud_cover",              "int"),
    ("cloud_cover_low",          "cloud_cover_low",          "int"),
    ("cloud_cover_mid",          "cloud_cover_mid",          "int"),
    ("cloud_cover_high",         "cloud_cover_high",         "int"),
    ("dew_point_2m",             "dew_point_2m",             "double"),
    ("is_day",                   "is_day",                   "boolean"),
    ("shortwave_radiation",      "shortwave_radiation",      "double"),
    ("direct_radiation",         "direct_radiation",         "double"),
    ("direct_normal_irradiance", "direct_normal_irradiance", "double"),
    ("diffuse_radiation",        "diffuse_radiation",        "double"),
    ("terrestrial_radiation",   "terrestrial_radiation",   "double"),
    ("sunshine_duration",        "sunshine_duration",        "double"),
    ("uv_index",                 "uv_index",                 "double"),
    ("uv_index_clear_sky",       "uv_index_clear_sky",       "double"),
    ("vapour_pressure_deficit",  "vapour_pressure_deficit",  "double"),
    ("visibility",               "visibility",               "double"),
    ("wet_bulb_temperature_2m",  "wet_bulb_temperature_2m",  "double"),
    ("wind_speed_10m",           "wind_speed_10m",           "double"),
    ("wind_speed_80m",           "wind_speed_80m",           "double"),
    ("wind_direction_10m",       "wind_direction_10m",       "int"),
    ("wind_direction_80m",       "wind_direction_80m",       "int"),
    ("wind_gusts_10m",           "wind_gusts_10m",           "double"),
    ("soil_temperature_0cm",     "soil_temperature_0cm",     "double"),
    ("soil_temperature_6cm",     "soil_temperature_6cm",     "double"),
    ("soil_temperature_18cm",    "soil_temperature_18cm",    "double"),
    ("soil_temperature_54cm",    "soil_temperature_54cm",    "double"),
    ("soil_moisture_0_to_1cm",   "soil_moisture_0_to_1cm",   "double"),
    ("soil_moisture_1_to_3cm",    "soil_moisture_1_to_3cm",   "double"),
    ("soil_moisture_3_to_9cm",    "soil_moisture_3_to_9cm",   "double"),
    ("soil_moisture_9_to_27cm",   "soil_moisture_9_to_27cm",   "double"),
    ("soil_moisture_27_to_81cm", "soil_moisture_27_to_81cm", "double"),
]

def _select_field(spec):
    """Build a select expression for one zipped field."""
    bronze_name, silver_name, dtype = spec
    base = col(f"hourly.{bronze_name}")
    if dtype == "timestamp":
        return to_timestamp(base).alias(silver_name)
    elif dtype == "boolean":
        return base.cast("boolean").alias(silver_name)
    elif dtype == "int":
        return base.cast("int").alias(silver_name)
    else:
        return base.cast("double").alias(silver_name)

hourly_zip_cols = [col(f"raw_response.hourly.{s[0]}") for s in HOURLY_SPECS]

silver_hourly = bronze.select(
    col("metadata.location_name"),
    _location_id_expr,
    col("metadata.latitude"),
    col("metadata.longitude"),
    col("raw_response.elevation").alias("elevation_m"),
    to_timestamp(col("metadata.retrieval_timestamp_utc")).alias("snapshot_at"),
    col("metadata.source"),
    col("metadata.api_endpoint"),
    col("_source_file_path").alias("source_file_path"),
    col("_ingest_timestamp").alias("ingested_at"),
    col("_rescued_data"),
    explode(arrays_zip(*hourly_zip_cols)).alias("hourly"),
).select(
    "location_name", "latitude", "longitude", "elevation_m",
    "snapshot_at", "source", "api_endpoint", "source_file_path",
    "ingested_at", "_rescued_data",
    *[_select_field(s) for s in HOURLY_SPECS],
)
hourly_count = silver_hourly.count()
print(f"Hourly rows: {hourly_count}")

# ─── Transform: Daily Forecast ─────────────────────────────────────
DAILY_SPECS = [
    ("time",                      "forecast_date",             "date"),
    ("temperature_2m_max",       "temperature_2m_max",       "double"),
    ("temperature_2m_min",       "temperature_2m_min",       "double"),
    ("apparent_temperature_max", "apparent_temperature_max", "double"),
    ("apparent_temperature_min", "apparent_temperature_min", "double"),
    ("sunrise",                   "sunrise",                   "timestamp"),
    ("sunset",                   "sunset",                   "timestamp"),
    ("daylight_duration",        "daylight_duration",        "double"),
    ("sunshine_duration",        "sunshine_duration",        "double"),
    ("uv_index_max",             "uv_index_max",             "double"),
    ("precipitation_sum",        "precipitation_sum",        "double"),
    ("rain_sum",                 "rain_sum",                 "double"),
    ("shortwave_radiation_sum",  "shortwave_radiation_sum",  "double"),
    ("snowfall_sum",             "snowfall_sum",             "double"),
    ("precipitation_hours",      "precipitation_hours",      "double"),
    ("precipitation_probability_max", "precipitation_probability_max", "int"),
    ("weather_code",             "weather_code",             "int"),
    ("wind_speed_10m_max",       "wind_speed_10m_max",       "double"),
    ("wind_gusts_10m_max",       "wind_gusts_10m_max",       "double"),
    ("wind_direction_10m_dominant", "wind_direction_10m_dominant", "int"),
    ("et0_fao_evapotranspiration", "et0_fao_evapotranspiration", "double"),
]

def _daily_select(spec):
    """Build a select expression for one zipped daily field."""
    bronze_name, silver_name, dtype = spec
    base = col(f"daily.{bronze_name}")
    if dtype == "date":
        return to_date(base).alias(silver_name)
    elif dtype == "timestamp":
        return to_timestamp(base).alias(silver_name)
    elif dtype == "int":
        return base.cast("int").alias(silver_name)
    else:
        return base.cast("double").alias(silver_name)

daily_zip_cols = [col(f"raw_response.daily.{s[0]}") for s in DAILY_SPECS]

silver_daily = bronze.select(
    col("metadata.location_name"),
    _location_id_expr,
    col("metadata.latitude"),
    col("metadata.longitude"),
    col("raw_response.elevation").alias("elevation_m"),
    to_timestamp(col("metadata.retrieval_timestamp_utc")).alias("snapshot_at"),
    col("metadata.source"),
    col("metadata.api_endpoint"),
    col("_source_file_path").alias("source_file_path"),
    col("_ingest_timestamp").alias("ingested_at"),
    col("_rescued_data"),
    explode(arrays_zip(*daily_zip_cols)).alias("daily"),
).select(
    "location_name", "latitude", "longitude", "elevation_m",
    "snapshot_at", "source", "api_endpoint", "source_file_path",
    "ingested_at", "_rescued_data",
    *[_daily_select(s) for s in DAILY_SPECS],
)
daily_count = silver_daily.count()
print(f"Daily rows: {daily_count}")

# ─── Validation ────────────────────────────────────────────────────
expected_hourly = total_bronze * 24
expected_daily = total_bronze * 1
print(f"\nExpected: current={total_bronze}, hourly={expected_hourly}, daily={expected_daily}")
print(f"Actual:   current={current_count}, hourly={hourly_count}, daily={daily_count}")
assert current_count == total_bronze, "Current count mismatch!"
assert hourly_count == expected_hourly, "Hourly count mismatch!"
assert daily_count == expected_daily, "Daily count mismatch!"

# Duplicate key checks
curr_dups = silver_current.groupBy("location_name", "snapshot_at").agg(cnt("*").alias("n")).filter(col("n") > 1)
print(f"Current duplicate keys: {curr_dups.count()}")
hrly_dups = silver_hourly.groupBy("location_name", "snapshot_at", "forecast_for").agg(cnt("*").alias("n")).filter(col("n") > 1)
print(f"Hourly duplicate keys: {hrly_dups.count()}")
dly_dups = silver_daily.groupBy("location_name", "snapshot_at", "forecast_date").agg(cnt("*").alias("n")).filter(col("n") > 1)
print(f"Daily duplicate keys: {dly_dups.count()}")

# Rescued data
rescued = bronze.filter(col("_rescued_data").isNotNull()).count()
print(f"\nRescued data records: {rescued}")

print("\n✓ Transform validation passed.")

In [0]:
# ════════════════════════════════════════════════════════════════════
# WEATHER SILVER — WRITE & VALIDATE
# ════════════════════════════════════════════════════════════════════

# ── Add location_id column to existing tables (backward-compatible) ──
for _tbl in [SILVER_CURRENT, SILVER_HOURLY, SILVER_DAILY]:
    if spark.catalog.tableExists(_tbl):
        _has_loc_id = spark.sql(f"DESCRIBE TABLE {_tbl}").filter("col_name = 'location_id'").count() > 0
        if not _has_loc_id:
            spark.sql(f"ALTER TABLE {_tbl} ADD COLUMN location_id BIGINT")
            print(f"Added location_id column to {_tbl}")

# ─── Write: weather_current ───────────────────────────────────────
# Schema fix: drop if column types are stale (bigint → int) or interval_seconds missing
if spark.catalog.tableExists(SILVER_CURRENT):
    _cols = spark.sql(f"DESCRIBE TABLE {SILVER_CURRENT}")
    _rh_type = _cols.filter("col_name = 'relative_humidity_2m'").collect()[0].data_type
    _has_interval = _cols.filter("col_name = 'interval_seconds'").count() > 0
    if _rh_type == "bigint" or not _has_interval:
        spark.sql(f"DROP TABLE {SILVER_CURRENT}")
        print(f"Dropped {SILVER_CURRENT} for type standardization + interval_seconds")

if spark.catalog.tableExists(SILVER_CURRENT):
    DeltaTable.forName(spark, SILVER_CURRENT).alias("t").merge(
        silver_current.alias("s"),
        "t.location_name = s.location_name AND t.snapshot_at = s.snapshot_at"
    ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()
    print(f"MERGE completed: {SILVER_CURRENT}")
else:
    silver_current.write.format("delta").saveAsTable(SILVER_CURRENT)
    print(f"Created: {SILVER_CURRENT}")

# ─── Write: weather_hourly ────────────────────────────────────────
# Schema fix: drop if is_day is int (old type, should be boolean)
if spark.catalog.tableExists(SILVER_HOURLY):
    _isday_type = spark.sql(f"DESCRIBE TABLE {SILVER_HOURLY}").filter("col_name = 'is_day'").collect()[0].data_type
    if _isday_type == "int":
        spark.sql(f"DROP TABLE {SILVER_HOURLY}")
        print(f"Dropped {SILVER_HOURLY} for type standardization (is_day int → boolean)")

if spark.catalog.tableExists(SILVER_HOURLY):
    DeltaTable.forName(spark, SILVER_HOURLY).alias("t").merge(
        silver_hourly.alias("s"),
        "t.location_name = s.location_name AND t.snapshot_at = s.snapshot_at "
        "AND t.forecast_for = s.forecast_for"
    ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()
    print(f"MERGE completed: {SILVER_HOURLY}")
else:
    silver_hourly.write.format("delta").saveAsTable(SILVER_HOURLY)
    print(f"Created: {SILVER_HOURLY}")

# ─── Write: weather_daily ─────────────────────────────────────────
# Schema fix: drop if shortwave_radiation_sum is missing
if spark.catalog.tableExists(SILVER_DAILY):
    _has_swr = spark.sql(f"DESCRIBE TABLE {SILVER_DAILY}").filter("col_name = 'shortwave_radiation_sum'").count() > 0
    if not _has_swr:
        spark.sql(f"DROP TABLE {SILVER_DAILY}")
        print(f"Dropped {SILVER_DAILY} to add shortwave_radiation_sum")

if spark.catalog.tableExists(SILVER_DAILY):
    DeltaTable.forName(spark, SILVER_DAILY).alias("t").merge(
        silver_daily.alias("s"),
        "t.location_name = s.location_name AND t.snapshot_at = s.snapshot_at "
        "AND t.forecast_date = s.forecast_date"
    ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()
    print(f"MERGE completed: {SILVER_DAILY}")
else:
    silver_daily.write.format("delta").saveAsTable(SILVER_DAILY)
    print(f"Created: {SILVER_DAILY}")

# ─── Post-Write Validation ─────────────────────────────────────────
print("\n" + "=" * 70)
print("POST-WRITE VALIDATION")
print("=" * 70)

# Current
curr = spark.table(SILVER_CURRENT)
curr_rows = curr.count()
curr_keys = curr.select(countDistinct("location_name", "snapshot_at")).collect()[0][0]
print(f"\n[{SILVER_CURRENT}]")
print(f"  Rows: {curr_rows}, Distinct keys: {curr_keys}")
assert curr_rows == curr_keys, "Duplicate keys in current!"
print("  Null counts:")
curr.agg(
    cnt("*").alias("total"),
    cnt("temperature_2m").alias("temp_nn"),
    cnt("relative_humidity_2m").alias("humidity_nn"),
    cnt("precipitation").alias("precip_nn"),
    cnt("wind_speed_10m").alias("wind_nn"),
    cnt("weather_code").alias("wcode_nn"),
    cnt("interval_seconds").alias("interval_nn"),
).show(truncate=False)
print("  Timestamp ranges:")
curr.agg(
    min_fn("snapshot_at").alias("min_snapshot"),
    max_fn("snapshot_at").alias("max_snapshot"),
    min_fn("observed_at").alias("min_observed"),
    max_fn("observed_at").alias("max_observed"),
).show(truncate=False)

# Hourly
hrly = spark.table(SILVER_HOURLY)
hrly_rows = hrly.count()
hrly_keys = hrly.select(countDistinct("location_name", "snapshot_at", "forecast_for")).collect()[0][0]
print(f"\n[{SILVER_HOURLY}]")
print(f"  Rows: {hrly_rows}, Distinct keys: {hrly_keys}")
assert hrly_rows == hrly_keys, "Duplicate keys in hourly!"
print("  Null counts:")
hrly.agg(
    cnt("*").alias("total"),
    cnt("forecast_for").alias("forecast_nn"),
    cnt("temperature_2m").alias("temp_nn"),
    cnt("precipitation").alias("precip_nn"),
    cnt("soil_moisture_0_to_1cm").alias("soil_nn"),
).show(truncate=False)
print("  Timestamp ranges:")
hrly.agg(
    min_fn("forecast_for").alias("min_forecast"),
    max_fn("forecast_for").alias("max_forecast"),
).show(truncate=False)

# Daily
dly = spark.table(SILVER_DAILY)
dly_rows = dly.count()
dly_keys = dly.select(countDistinct("location_name", "snapshot_at", "forecast_date")).collect()[0][0]
print(f"\n[{SILVER_DAILY}]")
print(f"  Rows: {dly_rows}, Distinct keys: {dly_keys}")
assert dly_rows == dly_keys, "Duplicate keys in daily!"
print("  Null counts:")
dly.agg(
    cnt("*").alias("total"),
    cnt("forecast_date").alias("date_nn"),
    cnt("temperature_2m_max").alias("tmax_nn"),
    cnt("precipitation_sum").alias("precip_nn"),
    cnt("sunrise").alias("sunrise_nn"),
    cnt("shortwave_radiation_sum").alias("swr_nn"),
).show(truncate=False)
print("  Date ranges:")
dly.agg(
    min_fn("forecast_date").alias("min_date"),
    max_fn("forecast_date").alias("max_date"),
).show(truncate=False)

# Reconciliation
print("\n--- Reconciliation ---")
print(f"  Bronze rows: {total_bronze}")
print(f"  Current rows: {curr_rows} (expected: {total_bronze})")
print(f"  Hourly rows: {hrly_rows} (expected: {total_bronze * 24})")
print(f"  Daily rows: {dly_rows} (expected: {total_bronze * 1})")
print(f"  Rescued data: {rescued}")
assert curr_rows == total_bronze, "Current count mismatch!"
assert hrly_rows == total_bronze * 24, "Hourly count mismatch!"
assert dly_rows == total_bronze * 1, "Daily count mismatch!"

# Sample rows
print("\n--- Sample current rows ---")
curr.select("location_name", "snapshot_at", "observed_at", "temperature_2m", "apparent_temperature", "relative_humidity_2m", "precipitation", "weather_code", "wind_speed_10m").show(5, truncate=False)

print("\n--- Sample hourly rows ---")
hrly.select("location_name", "snapshot_at", "forecast_for", "temperature_2m", "precipitation", "cloud_cover", "wind_speed_10m", "soil_temperature_0cm").orderBy("location_name", "forecast_for").show(5, truncate=False)

print("\n--- Sample daily rows ---")
dly.select("location_name", "snapshot_at", "forecast_date", "temperature_2m_max", "temperature_2m_min", "precipitation_sum", "sunrise", "sunset", "uv_index_max").show(5, truncate=False)

print("\n✓ Weather Silver layer complete.")

In [0]:
# ════════════════════════════════════════════════════════════════════
# WEATHER SILVER — COLUMN COMMENTS
# ════════════════════════════════════════════════════════════════════
# Idempotent: ALTER COLUMN COMMENT can be run multiple times safely.

_comments = {
    "weather_api.silver.weather_current": {
        "location_name": "City name used as the location identifier",
        "latitude": "Requested latitude coordinate",
        "longitude": "Requested longitude coordinate",
        "elevation_m": "Elevation of the location, in metres above sea level",
        "snapshot_at": "UTC timestamp when the source API snapshot was retrieved",
        "observed_at": "UTC timestamp the current weather reading applies to",
        "source": "Pipeline source label",
        "api_endpoint": "Full API URL called for this observation",
        "source_file_path": "UC Volume path of the source JSON file",
        "ingested_at": "Timestamp when Auto Loader ingested this record into Bronze",
        "_rescued_data": "Fields that did not match the Bronze schema (null = no schema drift)",
        "temperature_2m": "Air temperature at 2 metres above ground, in °C",
        "apparent_temperature": "Apparent ('feels like') temperature at 2m, in °C",
        "relative_humidity_2m": "Relative humidity at 2 metres, in % (0–100)",
        "is_day": "Day/night indicator (true = daytime, false = nighttime)",
        "precipitation": "Total precipitation for the current interval, in mm",
        "rain": "Rainfall for the current interval, in mm",
        "snowfall": "Snowfall for the current interval, in cm",
        "weather_code": "WMO weather interpretation code (numeric code, not a measurement)",
        "cloud_cover": "Total cloud cover, in % (0–100)",
        "pressure_msl": "Mean sea-level atmospheric pressure, in hPa",
        "surface_pressure": "Atmospheric pressure at the surface, in hPa",
        "wind_speed_10m": "Wind speed at 10 metres above ground, in km/h",
        "wind_direction_10m": "Wind direction at 10 metres, in degrees (0=N, 90=E, 180=S, 270=W)",
        "wind_gusts_10m": "Wind gusts at 10 metres above ground, in km/h",
        "interval_seconds": "Time interval the current reading represents, in seconds (3600 = hourly value)",
    },
    "weather_api.silver.weather_hourly": {
        "location_name": "City name used as the location identifier",
        "latitude": "Requested latitude coordinate",
        "longitude": "Requested longitude coordinate",
        "elevation_m": "Elevation of the location, in metres above sea level",
        "snapshot_at": "UTC timestamp when the source API snapshot was retrieved",
        "source": "Pipeline source label",
        "api_endpoint": "Full API URL called for this forecast",
        "source_file_path": "UC Volume path of the source JSON file",
        "ingested_at": "Timestamp when Auto Loader ingested this record into Bronze",
        "_rescued_data": "Fields that did not match the Bronze schema (null = no schema drift)",
        "forecast_for": "UTC timestamp for this forecast hour",
        "temperature_2m": "Air temperature at 2 metres above ground, in °C",
        "relative_humidity_2m": "Relative humidity at 2 metres, in % (0–100)",
        "apparent_temperature": "Apparent ('feels like') temperature at 2m, in °C",
        "precipitation": "Hourly precipitation, in mm",
        "rain": "Hourly rainfall, in mm",
        "snowfall": "Hourly snowfall, in cm",
        "snow_depth": "Snow depth on the ground, in m",
        "weather_code": "WMO weather interpretation code (numeric code, not a measurement)",
        "pressure_msl": "Mean sea-level atmospheric pressure, in hPa",
        "surface_pressure": "Atmospheric pressure at the surface, in hPa",
        "cloud_cover": "Total cloud cover, in % (0–100)",
        "cloud_cover_low": "Low-altitude cloud cover (below ~2 km), in % (0–100)",
        "cloud_cover_mid": "Mid-altitude cloud cover (2–6 km), in % (0–100)",
        "cloud_cover_high": "High-altitude cloud cover (above ~6 km), in % (0–100)",
        "dew_point_2m": "Dew point temperature at 2m, in °C",
        "is_day": "Day/night indicator (true = daytime, false = nighttime)",
        "shortwave_radiation": "Total solar radiation on horizontal surface, in W/m²",
        "direct_radiation": "Direct solar radiation on horizontal surface, in W/m²",
        "direct_normal_irradiance": "Direct solar radiation on a perpendicular surface, in W/m²",
        "diffuse_radiation": "Scattered solar radiation, in W/m²",
        "terrestrial_radiation": "Longwave thermal radiation from Earth's surface, in W/m²",
        "sunshine_duration": "Sunshine duration for this hour, in seconds",
        "uv_index": "Solar ultraviolet radiation level (0=none, 11+=extreme)",
        "uv_index_clear_sky": "UV index assuming clear sky conditions",
        "vapour_pressure_deficit": "Vapour pressure deficit, in kPa (drives evapotranspiration)",
        "visibility": "Horizontal visibility, in metres",
        "wet_bulb_temperature_2m": "Wet-bulb temperature at 2m, in °C (evaporative cooling limit)",
        "wind_speed_10m": "Wind speed at 10 metres above ground, in km/h",
        "wind_speed_80m": "Wind speed at 80 metres above ground, in km/h",
        "wind_direction_10m": "Wind direction at 10 metres, in degrees (0=N, 90=E, 180=S, 270=W)",
        "wind_direction_80m": "Wind direction at 80 metres, in degrees",
        "wind_gusts_10m": "Wind gusts at 10 metres above ground, in km/h",
        "soil_temperature_0cm": "Soil temperature at the surface, in °C",
        "soil_temperature_6cm": "Soil temperature at 6 cm depth, in °C",
        "soil_temperature_18cm": "Soil temperature at 18 cm depth, in °C",
        "soil_temperature_54cm": "Soil temperature at 54 cm depth, in °C",
        "soil_moisture_0_to_1cm": "Volumetric soil moisture for 0–1 cm layer, in m³/m³",
        "soil_moisture_1_to_3cm": "Volumetric soil moisture for 1–3 cm layer, in m³/m³",
        "soil_moisture_3_to_9cm": "Volumetric soil moisture for 3–9 cm layer, in m³/m³",
        "soil_moisture_9_to_27cm": "Volumetric soil moisture for 9–27 cm layer, in m³/m³",
        "soil_moisture_27_to_81cm": "Volumetric soil moisture for 27–81 cm layer, in m³/m³",
    },
    "weather_api.silver.weather_daily": {
        "location_name": "City name used as the location identifier",
        "latitude": "Requested latitude coordinate",
        "longitude": "Requested longitude coordinate",
        "elevation_m": "Elevation of the location, in metres above sea level",
        "snapshot_at": "UTC timestamp when the source API snapshot was retrieved",
        "source": "Pipeline source label",
        "api_endpoint": "Full API URL called for this forecast",
        "source_file_path": "UC Volume path of the source JSON file",
        "ingested_at": "Timestamp when Auto Loader ingested this record into Bronze",
        "_rescued_data": "Fields that did not match the Bronze schema (null = no schema drift)",
        "forecast_date": "Calendar date for this forecast day",
        "temperature_2m_max": "Maximum air temperature at 2m, in °C",
        "temperature_2m_min": "Minimum air temperature at 2m, in °C",
        "apparent_temperature_max": "Maximum apparent ('feels like') temperature, in °C",
        "apparent_temperature_min": "Minimum apparent ('feels like') temperature, in °C",
        "sunrise": "UTC timestamp of sunrise",
        "sunset": "UTC timestamp of sunset",
        "daylight_duration": "Total daylight time, in seconds",
        "sunshine_duration": "Total sunshine duration, in seconds",
        "uv_index_max": "Maximum UV index for the day",
        "precipitation_sum": "Total precipitation for the day, in mm",
        "rain_sum": "Total rainfall for the day, in mm",
        "shortwave_radiation_sum": "Total solar radiation received for the day, in MJ/m²",
        "snowfall_sum": "Total snowfall for the day, in cm",
        "precipitation_hours": "Number of hours with precipitation",
        "precipitation_probability_max": "Maximum precipitation probability, in % (0–100)",
        "weather_code": "WMO weather interpretation code (numeric code, not a measurement)",
        "wind_speed_10m_max": "Maximum sustained wind speed at 10m, in km/h",
        "wind_gusts_10m_max": "Maximum wind gusts at 10m, in km/h",
        "wind_direction_10m_dominant": "Dominant wind direction at 10m, in degrees",
        "et0_fao_evapotranspiration": "FAO reference evapotranspiration, in mm",
    },
}

_comment_count = 0
for _table, _col_comments in _comments.items():
    for _col, _comment in _col_comments.items():
        _escaped = _comment.replace("'", "''")
        spark.sql(f"ALTER TABLE {_table} ALTER COLUMN {_col} COMMENT '{_escaped}'")
        _comment_count += 1
print(f"Applied {_comment_count} column comments across {len(_comments)} weather Silver tables.")